# Lab 3: Multi-Source Retail Sales Data Integration and Analysis

**Domain:** Retail Analytics &nbsp;|&nbsp; **Platform:** R (Google Colab, R runtime) &nbsp;|&nbsp; **Duration:** 2 Hours

**Dataset:** [UCI Machine Learning Repository — Online Retail Dataset](https://archive.ics.uci.edu/dataset/352/online+retail)
(real dataset, 541,909 transaction-level rows, UK-based online retailer, Dec 2010 – Dec 2011)

A multinational retail company keeps its data in three separate systems — sales transactions (CSV),
product information (JSON), and customer information (Excel). This notebook imports, cleans, integrates
and analyzes these heterogeneous sources in R, then persists the final dataset in a SQLite database.

**Source files** (in `data/`, derived from the real UCI dataset):
- `transactions.csv` — InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate
- `products.json` — StockCode, Description, UnitPrice
- `customers.xlsx` — CustomerID, Country


## Setup: Packages

In [1]:
# Run once per Colab session (R runtime). Colab's R image already ships
# most of these; install.packages() is a no-op if already present.
packages <- c("readr", "jsonlite", "readxl", "dplyr", "DBI", "RSQLite", "tidyr")
new_pkgs <- packages[!(packages %in% installed.packages()[, "Package"])]
if (length(new_pkgs) > 0) install.packages(new_pkgs)

library(readr)
library(jsonlite)
library(readxl)
library(dplyr)
library(DBI)
library(RSQLite)
library(tidyr)


Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, union


## Task 1: Import and Clean the Data

Import the three heterogeneous sources, inspect them, and clean:
- missing values
- duplicate records
- invalid/zero quantities
- invalid/zero unit prices

then derive `Revenue = Quantity * UnitPrice`.


In [2]:
transactions <- read_csv("data/transactions.csv",
                          col_types = cols(
                            InvoiceNo   = col_character(),
                            StockCode   = col_character(),   # source Excel stores some
                                                              # stock codes as numbers and
                                                              # others as text -- force
                                                              # character so joins in Task 2
                                                              # aren't silently broken
                            CustomerID  = col_double(),
                            Quantity    = col_double(),
                            InvoiceDate = col_character()
                          ))

products  <- fromJSON("data/products.json")
customers <- read_excel("data/customers.xlsx")

cat("transactions:", dim(transactions), "\n")
cat("products    :", dim(products), "\n")
cat("customers   :", dim(customers), "\n")


transactions: 541909 5
products    : 4070 3
customers   : 4372 2


In [3]:
# Missing values per column
cat("---- transactions ----\n"); print(colSums(is.na(transactions)))
cat("---- products ----\n");     print(colSums(is.na(products)))
cat("---- customers ----\n");    print(colSums(is.na(customers)))


---- transactions ----
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate
          0           0      135080           0           0
---- products ----
  StockCode Description   UnitPrice
          0         112           0
---- customers ----
CustomerID    Country
         0          0


In [4]:
dup_count <- sum(duplicated(transactions))
non_positive_qty <- sum(transactions$Quantity <= 0, na.rm = TRUE)
cat("Duplicate transaction rows:", dup_count, "\n")
cat("Non-positive Quantity rows:", non_positive_qty, "\n")


Duplicate transaction rows: 5429
Non-positive Quantity rows: 10624


### Cleaning decisions

| Issue | Decision | Reason |
|---|---|---|
| Missing `CustomerID` (135,080 rows, ~25%) | Dropped | Cannot attribute a transaction to a customer for the customer-value analysis in Task 3; these are largely bulk/guest orders |
| Duplicate transaction rows (5,429) | Dropped (`distinct()`) | Same invoice/product/customer/quantity/timestamp repeated — data entry duplication, not repeat purchases |
| `Quantity <= 0` (10,624 rows) | Dropped | Negative quantities are cancellations (`InvoiceNo` starting with `C`); zero quantities are not real sales |
| Missing/zero `UnitPrice` or `Description` in products | Dropped from the product table | Cannot compute `Revenue` without a valid price, and a product with no description is not usable in the top-products report |
| Missing `Country` in customers | Dropped | Cannot attribute revenue to a market without a country |


In [5]:
transactions_clean <- transactions %>%
  distinct() %>%
  filter(!is.na(CustomerID), Quantity > 0) %>%
  mutate(CustomerID = as.integer(CustomerID))

products_clean <- products %>%
  filter(!is.na(Description), !is.na(UnitPrice), UnitPrice > 0)

customers_clean <- customers %>%
  filter(!is.na(Country))

cat("transactions_clean:", dim(transactions_clean), "\n")
cat("products_clean    :", dim(products_clean), "\n")
cat("customers_clean   :", dim(customers_clean), "\n")


transactions_clean: 392708 5
products_clean    : 3846 3
customers_clean   : 4372 2


## Task 2: Integrate the Multiple Data Sources

`transactions_clean` is joined to `products_clean` on `StockCode`, then to `customers_clean` on
`CustomerID`, both using `left_join()`.

**Why `left_join()` and not `inner_join()`:** the transaction table is the fact table we care about —
every cleaned sale should be kept even if a lookup happens to be missing, so unmatched rows can be
surfaced and inspected rather than silently discarded by an `inner_join()`. Rows still missing a
price or a country after both joins (unmatched lookups) are then dropped explicitly, which is
equivalent to an inner join in the end but makes the unmatched count visible first.


In [6]:
merged <- transactions_clean %>%
  left_join(products_clean, by = "StockCode")

unmatched_products <- sum(is.na(merged$UnitPrice))
cat("Rows after product left_join:", nrow(merged), "\n")
cat("Unmatched product rows:", unmatched_products, "\n")

merged <- merged %>%
  left_join(customers_clean, by = "CustomerID")

unmatched_customers <- sum(is.na(merged$Country))
cat("Rows after customer left_join:", nrow(merged), "\n")
cat("Unmatched customer rows:", unmatched_customers, "\n")


Rows after product left_join: 392708
Unmatched product rows: 92
Rows after customer left_join: 392708
Unmatched customer rows: 0


In [7]:
retail_sales <- merged %>%
  filter(!is.na(UnitPrice), !is.na(Country)) %>%
  mutate(Revenue = Quantity * UnitPrice)

dim(retail_sales)
head(retail_sales, 3)


[1] 392616      9


  InvoiceNo StockCode CustomerID Quantity         InvoiceDate                         Description UnitPrice        Country Revenue
1    536365    85123A      17850        6 2010-12-01 08:26:00 WHITE HANGING HEART T-LIGHT HOLDER      2.95 United Kingdom    17.7
2    536365     71053      17850        6 2010-12-01 08:26:00                WHITE METAL LANTERN      3.75 United Kingdom    22.5
3    536365    84406B      17850        8 2010-12-01 08:26:00     CREAM CUPID HEARTS COAT HANGER      4.15 United Kingdom    33.2

Only **92 of 392,708** rows (0.02%) fail the product join — a handful of stock codes (adjustment /
sample codes) that never had a valid positive price anywhere in the source data, so they were
correctly excluded by the `products_clean` filter in Task 1. Every customer matched (`customers_clean`
was built to cover exactly the customer IDs present in the transaction data), so the customer join is
a perfect match. The final integrated dataset has **392,616 rows and 9 columns**.


## Task 3: Sales and Customer Analysis

### 1. Total sales revenue

In [8]:
total_revenue <- retail_sales %>% summarise(Total = sum(Revenue)) %>% pull(Total)
cat("Total sales revenue: £", format(round(total_revenue, 2), big.mark = ","), "\n", sep = "")


Total sales revenue: £9,581,039.28


### 2. Top 5 products by revenue

In [9]:
top5_products <- retail_sales %>%
  group_by(StockCode, Description) %>%
  summarise(Revenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(Revenue)) %>%
  head(5)

top5_products


# A tibble: 5 x 3
  StockCode Description                          Revenue
  <chr>     <chr>                                  <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE          168470.
2 22423     REGENCY CAKESTAND 3 TIER             157896.
3 85123A    WHITE HANGING HEART T-LIGHT HOLDER   108451.
4 23166     MEDIUM CERAMIC TOP STORAGE JAR        97395.
5 85099B    JUMBO BAG RED RETROSPOT               95842.

### 3. Top 5 countries by revenue

In [10]:
top5_countries <- retail_sales %>%
  group_by(Country) %>%
  summarise(Revenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(Revenue)) %>%
  head(5)

top5_countries


# A tibble: 5 x 2
  Country          Revenue
  <chr>              <dbl>
1 United Kingdom  7889728.
2 Netherlands       336485.
3 EIRE              286622.
4 Germany           235546.
5 France            206117.

### 4. Top 5 customers by total purchase value

In [11]:
top5_customers <- retail_sales %>%
  group_by(CustomerID) %>%
  summarise(Revenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(Revenue)) %>%
  head(5)

top5_customers


# A tibble: 5 x 2
  CustomerID Revenue
       <int>   <dbl>
1      18102 404250.
2      14646 331127.
3      17450 179187.
4      16446 168473.
5      14911 150923.

### Customer value segmentation

Thresholds are the 50th/75th percentile and 3x the 75th percentile of per-customer total revenue,
so the segments reflect the actual shape of this customer base rather than arbitrary fixed numbers.


In [12]:
customer_revenue <- retail_sales %>%
  group_by(CustomerID) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop")

q50 <- quantile(customer_revenue$TotalRevenue, 0.50)
q75 <- quantile(customer_revenue$TotalRevenue, 0.75)
cat("Median customer revenue (q50):", round(q50, 2), "\n")
cat("75th percentile (q75):", round(q75, 2), "\n")
cat("Premium threshold (3 x q75):", round(3 * q75, 2), "\n")

customer_segments <- customer_revenue %>%
  mutate(Segment = case_when(
    TotalRevenue >= 3 * q75 ~ "Premium",
    TotalRevenue >= q75     ~ "High Value",
    TotalRevenue >= q50     ~ "Medium Value",
    TRUE                    ~ "Low Value"
  ))

customer_segments %>% count(Segment, sort = TRUE)


Median customer revenue (q50): 689 
75th percentile (q75): 1721.83 
Premium threshold (3 x q75): 5165.49 


# A tibble: 4 x 2
  Segment          n
  <chr>        <int>
1 Low Value     2169
2 Medium Value  1085
3 High Value     806
4 Premium        279

### High-performing vs. underperforming market

- **High-performing market: United Kingdom.** It generates **£7,889,728** — about **82%** of total
  revenue — consistent with this being a UK-based online retailer with its largest, most established
  customer base concentrated domestically.
- **Underperforming market: Saudi Arabia** (and similarly Bahrain, Czech Republic). Saudi Arabia
  contributes only **£162.96** in total revenue, the lowest of any country present, suggesting minimal
  market penetration or only a handful of one-off orders rather than any sustained customer base —
  a candidate for either a dedicated market-entry push or de-prioritization.


## Task 4: Store and Retrieve Data Using SQL

Export `retail_sales` to a SQLite database and run two queries directly from R via `DBI`/`RSQLite`.


In [13]:
con <- dbConnect(RSQLite::SQLite(), "data/retail_sales.db")
dbWriteTable(con, "retail_sales", retail_sales, overwrite = TRUE)
dbListTables(con)


[1] "retail_sales"

In [14]:
# Query 1: Top 5 customers by revenue
query1 <- dbGetQuery(con, "
  SELECT CustomerID, ROUND(SUM(Revenue), 2) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5;
")
query1


  CustomerID TotalRevenue
1      18102    404250.20
2      14646    331127.47
3      17450    179187.37
4      16446    168472.50
5      14911    150922.66

In [15]:
# Query 2: Total revenue by country
query2 <- dbGetQuery(con, "
  SELECT Country, ROUND(SUM(Revenue), 2) AS TotalRevenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY TotalRevenue DESC;
")
head(query2, 10)

dbDisconnect(con)


          Country TotalRevenue
1  United Kingdom   7889728.20
2     Netherlands    336484.63
3            EIRE    286621.79
4         Germany    235545.83
5          France    206117.30
6       Australia    160788.11
7           Spain     62935.93
8     Switzerland     57292.39
9         Belgium     42982.47
10          Japan     42232.52

## Business Insights

1. **Revenue is heavily concentrated in a single market and a small customer base.** The UK accounts
   for ~82% of the £9.58M total revenue, and the top 5 customers alone contribute over £1.23M
   (~13% of total revenue). This concentration is a growth opportunity (international expansion is
   largely untapped) but also a risk (over-reliance on one geography and a handful of accounts).

2. **A small number of SKUs drive disproportionate revenue.** The top 5 products (`23843`, `22423`,
   `85123A`, `23166`, `85099B`) alone generate over £627,000. Stock and marketing priority should
   protect availability of these lines, and their bundling/cross-sell potential is worth testing.

3. **Only 279 of 4,339 customers (≈6.4%) are "Premium," but they likely account for a large share of
   profit.** A loyalty or account-management program targeted at the High Value / Premium segments
   (1,085 combined) is a more efficient growth lever than broad-based acquisition, given how thin the
   underperforming international markets (Saudi Arabia, Bahrain, Czech Republic) currently are.
